# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahaamjad2004/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
* **Unit of Analysis (Grain):** One unique published URL / page evaluated over a 30-day performance window.
* **Time Window:** Mid-panel evaluation month `2026-03` (historical observation window vs. 90-day decay horizon).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

# Repository path environment setup
if "google.colab" in sys.modules:
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# Load starter panel data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify grain: Unique URLs
print(f"Total Rows in Dataset: {len(df):,}")
print(f"Unique URLs: {df['url'].nunique():,}" if 'url' in df.columns else "Each row represents a unique page evaluation.")


Total Rows in Dataset: 30,000
Each row represents a unique page evaluation.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
* **Features:** `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr` (knowable at decision moment from historical logs).
* **Label / Proxy:** `target_declining = (trend_direction == 'down')` (binary decay indicator).
* **Context:** `url`, `category`, `primary_keyword` (identifiers and metadata).
* **Excluded:** `trend_pct` — **Why:** It directly contains the raw target math (`trend_direction` derivation) and causes total outcome leakage if included during training.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Construct target proxy label
df["target_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Bucket verification
feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
excluded_cols = ["trend_pct"]

print("Feature Set Shape:", df[feature_cols].shape)
print("Target Class Balance:\n", df["target_declining"].value_counts(normalize=True).round(3))


Feature Set Shape: (30000, 5)
Target Class Balance:
 target_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
* **Query 1 (Grain):** Proves 1 row = 1 unique page instance.
* **Query 2 (Availability):** Filters `IS TRUE` for complete signals.
* **Query 3 (Leakage Experiment):** Trains honest vs. leaky model (`trend_pct`) to demonstrate trap removal.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Grain Check
print("--- Query 1: Grain Check ---")
print(f"Total evaluated pages: {len(df):,}")

# 2. Availability Filter (IS TRUE)
df["is_available"] = (df["impressions_90d"] > 0) & (df["days_since_last_update"].notna())
surviving = df["is_available"].sum()
print(f"\n--- Query 2: Availability Check (IS TRUE) ---")
print(f"Surviving Rows: {surviving:,} / {len(df):,} ({surviving/len(df):.1%})")

# 3. Honest vs Leaky Model Demonstration
X_honest = df[feature_cols].fillna(0)
y = df["target_declining"]

clf_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
clf_honest.fit(X_honest, y)
honest_prec = precision_score(y, clf_honest.predict(X_honest))

X_leaky = X_honest.copy()
X_leaky["leaky_trend_pct"] = df["trend_pct"]

clf_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
clf_leaky.fit(X_leaky, y)
leaky_prec = precision_score(y, clf_leaky.predict(X_leaky))

print(f"\n--- Query 3: Leakage Trap Experiment ---")
print(f"Honest Model Precision: {honest_prec:.3f}")
print(f"Leaky Model Precision (with trend_pct): {leaky_prec:.3f} <-- Trap Triggered!")

# Trap Removal
del X_leaky
print("Leaky column purged. Honest feature set retained.")


--- Query 1: Grain Check ---
Total evaluated pages: 30,000

--- Query 2: Availability Check (IS TRUE) ---
Surviving Rows: 30,000 / 30,000 (100.0%)

--- Query 3: Leakage Trap Experiment ---
Honest Model Precision: 0.651
Leaky Model Precision (with trend_pct): 1.000 <-- Trap Triggered!
Leaky column purged. Honest feature set retained.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
* **Observed Window Limitation:** A static single-month snapshot (`2026-03`) cannot account for annual seasonal shifts (e.g., Q4 e-commerce traffic surges).
* **Positional Aggregation:** Averaged position figures mask page-level ranking fluctuations across dynamic search SERP layouts.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summary statistics confirming dataset bounds
print("Dataset Span & Signal Ranges:")
print(df[feature_cols].describe().round(2))


Dataset Span & Signal Ranges:
       content_age_days  days_since_last_update  impressions_90d  \
count          30000.00                30000.00         30000.00   
mean             256.17                   46.10          5200.37   
std              132.71                   42.08         16838.02   
min               90.00                    1.00             1.00   
25%              132.00                   20.00            81.00   
50%              236.00                   20.00           731.00   
75%              333.00                  104.00          3615.25   
max              564.00                  373.00        517715.00   

       avg_position       ctr  
count      30000.00  30000.00  
mean          16.34      0.51  
std           15.22      3.28  
min            0.00      0.00  
25%            6.20      0.00  
50%           10.80      0.07  
75%           22.30      0.29  
max          245.00    100.00  


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.